# Stage 1 – AlphaFold3 Structure Analysis

This notebook provides interactive exploration of AlphaFold3 structure predictions for **SCN1A** and **SCN2A**.

## Sections
1. Load predicted structures
2. Visualise per-residue pLDDT confidence
3. Compare structure quality across proteins
4. Explore Na⁺ ion interaction sites
5. Save publication-quality figures

**Prerequisites** – run stage1 scripts first:
```
python stage1_alphafold/01_prepare_sequences.py
python stage1_alphafold/02_run_alphafold3.py   # requires AF3 credentials
python stage1_alphafold/03_analyze_structures.py
python stage1_alphafold/04_protein_ion_interaction.py
```

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add project root to path
sys.path.insert(0, os.path.abspath('../..'))
from utils.structure_utils import parse_pdb_atoms, extract_ca_coords
from utils.visualization import plot_plddt

RESULTS_DIR = '../../data/results/stage1'
RAW_DIR = '../../data/raw'
print('Imports OK')

## 1. Load pLDDT Scores

pLDDT (predicted Local Distance Difference Test) scores indicate per-residue model confidence.
Scores are stored in the B-factor column of AlphaFold3 CIF/PDB outputs, or in confidence JSON files.

In [ ]:
def load_plddt_from_json(json_path: str) -> np.ndarray:
    """Load pLDDT from an AlphaFold3 confidence JSON file."""
    with open(json_path) as f:
        data = json.load(f)
    if 'plddt' in data:
        return np.array(data['plddt'])
    elif 'atom_plddts' in data:
        return np.array(data['atom_plddts'])
    raise ValueError(f'No pLDDT key in {json_path}')

def load_plddt_from_pdb(pdb_path: str) -> np.ndarray:
    """Load per-CA pLDDT from B-factor column of a PDB file."""
    from utils.structure_utils import parse_pdb_atoms
    atoms = parse_pdb_atoms(pdb_path)
    return np.array([a.b_factor for a in atoms if a.atom_name == 'CA'])

# --- Example usage ---
example_pdb = os.path.join(RAW_DIR, 'structures', 'scn1a_alphafold3_model.pdb')
example_json = os.path.join(RAW_DIR, 'structures', 'scn1a_alphafold3_confidence.json')

# Mock data if real outputs not available yet
if os.path.exists(example_json):
    plddt_scn1a = load_plddt_from_json(example_json)
elif os.path.exists(example_pdb):
    plddt_scn1a = load_plddt_from_pdb(example_pdb)
else:
    print('Note: AF3 output files not found — using mock pLDDT for demonstration')
    rng = np.random.default_rng(42)
    plddt_scn1a = np.clip(rng.normal(75, 18, 2009), 0, 100)

print(f'SCN1A pLDDT: {len(plddt_scn1a)} residues, mean={plddt_scn1a.mean():.1f}')

## 2. Visualise pLDDT

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(18, 8))
for ax, (gene, plddt) in zip(axes, [
    ('SCN1A', plddt_scn1a),
    ('SCN2A', np.roll(plddt_scn1a, 50)),  # placeholder for SCN2A
]):
    residues = np.arange(1, len(plddt) + 1)
    colors = np.where(plddt >= 90, '#1565C0',
              np.where(plddt >= 70, '#43A047',
              np.where(plddt >= 50, '#FFB300', '#E53935')))
    ax.bar(residues, plddt, color=colors, width=1, linewidth=0)
    for thresh, c in [(90, '#1565C0'), (70, '#43A047'), (50, '#FFB300')]:
        ax.axhline(thresh, color=c, lw=0.7, ls='--', alpha=0.5)
    ax.set_xlim(0, len(plddt) + 1)
    ax.set_ylim(0, 100)
    ax.set_xlabel('Residue')
    ax.set_ylabel('pLDDT')
    ax.set_title(f'{gene} pLDDT (mean = {plddt.mean():.1f})')

plt.tight_layout()
plt.show()

## 3. Summary Statistics

In [ ]:
def plddt_stats(name: str, plddt: np.ndarray) -> dict:
    return {
        'Gene': name,
        'Residues': len(plddt),
        'Mean pLDDT': round(plddt.mean(), 2),
        'Median pLDDT': round(np.median(plddt), 2),
        'Frac ≥90': round((plddt >= 90).mean(), 3),
        'Frac ≥70': round((plddt >= 70).mean(), 3),
        'Frac <50': round((plddt < 50).mean(), 3),
    }

df_stats = pd.DataFrame([
    plddt_stats('SCN1A', plddt_scn1a),
    plddt_stats('SCN2A', np.roll(plddt_scn1a, 50)),  # replace with real SCN2A
])
df_stats

## 4. Na⁺ Ion Binding Sites

Identify residues within 4 Å of modelled Na⁺ ions.

In [ ]:
ion_results_path = os.path.join(RESULTS_DIR, 'ion_binding_sites.csv')

if os.path.exists(ion_results_path):
    df_ions = pd.read_csv(ion_results_path)
    print(f'Loaded {len(df_ions)} ion binding interactions')
    display(df_ions.head(20))
else:
    print('Ion binding results not found — run stage1/04_protein_ion_interaction.py first')
    # Mock display
    df_ions = pd.DataFrame({
        'residue': [178, 179, 333, 334, 900, 901],
        'res_name': ['ASP', 'GLU', 'ASP', 'GLU', 'ASP', 'ASN'],
        'chain': ['A']*6,
        'distance_A': [2.4, 2.8, 3.1, 2.6, 3.3, 3.9],
        'ion_id': ['B', 'B', 'C', 'C', 'D', 'D'],
    })
    display(df_ions)